# Frameworks 02 - AWS Strands

Objetivo: ejecutar `strands.Agent` real sobre el Provider seleccionado por
`.env`, producir una respuesta humana con evidencia de Tool y demostrar por
separado hooks, structured output, sync/async y MCP.

**Lugar en el modelo:** Strands controla lifecycle, hooks, Tools y MCP; Bedrock
es solo uno de los Providers posibles.

**Evidencia exigida:** `human_result` debe registrar Provider, Framework,
modelo, Tool, respuesta publica y linaje. Las capacidades nativas adicionales
deben conservar evidencia verificable propia.

**Límite de la evidencia:** `python-runtime` solo certifica controles
deterministas del SDK y transports MCP. Una prueba live debe observar un
Provider LM real y una respuesta humana, no un schema tecnico.

## Parametros de la demostracion

`.env` es la fuente canonica. `AGENTIC_SYSTEMS_PROVIDER` puede fijar el Provider
o dejar `auto`; el flag live del Provider seleccionado autoriza la llamada.
`RUN_STRANDS_LIVE` es solamente un override opcional.

| Variable | Default | Proposito |
|---|---|---|
| AGENTIC_SYSTEMS_PROVIDER | auto | Provider utilizado por Strands. |
| RUN_STRANDS_LIVE | flag del Provider | Override opcional. |
| Framework | strands | Lifecycle, hooks, Tools y MCP reales. |

## 1) Provider x Framework y lifecycle


In [ ]:
import os

from pydantic import BaseModel
from strands import tool as strands_tool
from strands.hooks import AfterInvocationEvent

import agentic_systems as toolkit


def _enabled(value: str | None) -> bool:
    return str(value or "").strip().lower() in {"1", "true", "yes"}


requested_provider = os.getenv("AGENTIC_SYSTEMS_PROVIDER", "auto").strip() or "auto"
candidate_runtime = toolkit.runtime(provider=requested_provider)
candidate_description = candidate_runtime.describe()
selected_provider = candidate_description["selected_provider"]
provider_live_flag = (
    f"RUN_{selected_provider.removesuffix('-runtime').replace('-', '_').upper()}_LIVE"
    if selected_provider and selected_provider != "python-runtime"
    else None
)
provider_live_value = os.getenv(provider_live_flag, "0") if provider_live_flag else "0"
RUN_LIVE = (
    selected_provider != "python-runtime"
    and _enabled(os.getenv("RUN_STRANDS_LIVE", provider_live_value))
)
runtime = candidate_runtime if RUN_LIVE else toolkit.runtime(provider="python-runtime")
runtime_description = runtime.describe()
execution_kind = "live-language-model" if RUN_LIVE else "offline-deterministic-control"

profile = toolkit.integrations.framework_profile("strands")
toolkit.show_json(
    {
        "execution_kind": execution_kind,
        "requested_provider": requested_provider,
        "provider_live_flag": provider_live_flag,
        "provider_live_authorized": RUN_LIVE,
        "runtime": runtime_description,
        "profile": profile.to_dict(),
    },
    title="Provider x Strands preflight",
)


## 2) Tools mixtas, hook, structured output y sync/async


In [ ]:
class PublicEvidence(BaseModel):
    symbol: str
    is_public: bool


hook_events = []


def record_after_invocation(event: AfterInvocationEvent):
    hook_events.append(type(event).__name__)


@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}


@strands_tool
def native_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}


framework = toolkit.framework(
    "strands",
    agent_kwargs={"hooks": [record_after_invocation]},
)
agent = toolkit.agent(
    name="strands_inspector",
    instructions=(
        "Usa inspect_public_api para verificar el simbolo solicitado. "
        "Despues responde una sola frase natural en espanol; no devuelvas JSON."
    ),
    runtime=runtime,
    tools=[inspect_public_api],
    framework=framework,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
    policy=toolkit.RunPolicy(max_turns=4, max_tool_calls=1),
)
agent.prepare()
sync_request = (
    "Verifica si framework pertenece a la API publica instalada."
    if RUN_LIVE
    else {"tool": "inspect_public_api", "input": {"symbol": "framework"}}
)
async_request = (
    "Verifica si Skill pertenece a la API publica instalada."
    if RUN_LIVE
    else {"tool": "inspect_public_api", "input": {"symbol": "Skill"}}
)
sync_result = agent.run(sync_request, mode="eval")
async_result = await agent.arun(async_request, mode="eval")
assert sync_result.ok and async_result.ok
assert sync_result.engine == async_result.engine == runtime_description["selected_provider"]
assert sync_result.meta["framework_adapter"] == "strands"
assert async_result.meta["framework_adapter"] == "strands"
assert hook_events == ["AfterInvocationEvent", "AfterInvocationEvent"]
for observed in (sync_result, async_result):
    observed.raise_if_inconsistent()
    assert not observed.meta.get("fallback_provider")
    assert any(event.name == "inspect_public_api" and event.ok for event in observed.tool_events)
if RUN_LIVE:
    assert sync_result.engine != "python-runtime"
    assert sync_result.text and not sync_result.text.lstrip().startswith("{"), sync_result.text
    assert async_result.text and not async_result.text.lstrip().startswith("{"), async_result.text

toolkit.human_result(
    sync_result,
    title="Strands live RunResult" if RUN_LIVE else "Strands offline deterministic control",
    show_lineage=True,
)

# Structured output and native Strands Tool are deterministic SDK controls.
structured_agent = toolkit.agent(
    name="strands_structured_control",
    instructions="Devuelve PublicEvidence.",
    runtime=toolkit.runtime(provider="python-runtime"),
    tools=[inspect_public_api],
    framework=toolkit.framework(
        "strands",
        agent_kwargs={"tools": [native_public_api]},
        run_kwargs={"structured_output_model": PublicEvidence},
    ),
)
structured_result = structured_agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": "framework"}},
    mode="eval",
)
assert structured_result.ok
assert structured_result.engine == "python-runtime"
assert structured_result.meta["framework_adapter"] == "strands"
toolkit.show_json(
    {
        "execution_kind": execution_kind,
        "native_agent": type(agent.native_agent).__name__,
        "native_result": type(sync_result.native_result).__name__,
        "hook_events": hook_events,
        "async_text": async_result.text,
        "structured_control": structured_result.data,
    },
    title="Strands lifecycle evidence",
)


## 3) MCP local por stdio y Streamable HTTP


In [ ]:
from collections.abc import AsyncIterator, Callable
from contextlib import AbstractAsyncContextManager, asynccontextmanager
from functools import partial
from pathlib import Path
import os
import socket
import subprocess
import sys
import time

from mcp.client.stdio import StdioServerParameters, stdio_client
from mcp.client.streamable_http import streamable_http_client
from strands.tools.mcp import MCPClient

def locate_repo_file(relative: str) -> Path:
    for root in (Path.cwd(), *Path.cwd().parents):
        candidate = (root / relative).resolve()
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {relative!r} from kernel cwd {Path.cwd()}."
    )

SERVER = locate_repo_file("tutorials/frameworks/mcp_echo_server.py")

def free_port() -> int:
    with socket.socket() as listener:
        listener.bind(("127.0.0.1", 0))
        return int(listener.getsockname()[1])

def wait_for_port(process, port: int) -> None:
    deadline = time.monotonic() + 15
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError("The local MCP server exited early.")
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=0.2):
                return
        except OSError:
            time.sleep(0.05)
    raise TimeoutError("The local MCP server did not start.")

@asynccontextmanager
async def stdio_transport() -> AsyncIterator:
    parameters = StdioServerParameters(
        command=sys.executable,
        args=[str(SERVER), "--transport", "stdio"],
    )
    with open(os.devnull, "w", encoding="utf-8") as errlog:
        async with stdio_client(parameters, errlog=errlog) as transport:
            yield transport

def run_mcp_transport(transport: str):
    process = None
    transport_factory: Callable[[], AbstractAsyncContextManager]
    if transport == "stdio":
        transport_factory = stdio_transport
    else:
        port = free_port()
        process = subprocess.Popen(
            [sys.executable, str(SERVER), "--transport", "streamable-http", "--port", str(port)],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
        wait_for_port(process, port)
        transport_factory = partial(streamable_http_client, f"http://127.0.0.1:{port}/mcp")

    client = MCPClient(transport_factory)
    try:
        mcp_agent = toolkit.agent(
            name=f"strands_mcp_{transport}",
            instructions="Ejecuta la Tool MCP solicitada.",
            runtime=toolkit.runtime(provider="python-runtime"),
            framework=toolkit.framework("strands", agent_kwargs={"tools": [client]}),
        )
        return mcp_agent.run(
            {"tool": "echo", "input": {"value": "verified"}},
            mode="eval",
        )
    finally:
        client.stop(None, None, None)
        if process is not None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=5)

mcp_results = {
    transport: run_mcp_transport(transport)
    for transport in ("stdio", "streamable-http")
}
assert all(result.ok for result in mcp_results.values())
toolkit.show_json(
    {
        transport: {
            "data": result.data,
            "tool_events": [event.name for event in result.tool_events],
        }
        for transport, result in mcp_results.items()
    },
    title="Native Strands MCP",
)


## 4) API realmente ejercitada


In [ ]:
api_coverage = [
    "toolkit.runtime", "toolkit.framework", "toolkit.integrations.framework_profile",
    "toolkit.tool", "toolkit.agent", "Agent.prepare", "Agent.native_agent",
    "agent.run", "agent.arun", "RunResult.native_result", "toolkit.AgentContract", "toolkit.RunPolicy",
    "toolkit.human_result", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Strands API coverage")


## Resultado e interpretacion

Cuando el preflight declara `live-language-model`, el Provider seleccionado por
`.env` realiza inferencia y Strands controla lifecycle, hooks y Tools. La
respuesta publica debe ser humana y su linaje debe contener la Tool observada.

Structured output, la Tool nativa y los transports MCP se prueban como controles
separados. En `offline-deterministic-control`, `python-runtime` no se presenta
como modelo de lenguaje.